# Step-by-step Notebook Runner (based on `fyp_finance_ml_v2` code)

This notebook recreates your teammate's modular pipeline in notebook blocks so you can run and debug each stage interactively.

It is intentionally outside `fyp_finance_ml_v2/` as requested.


In [ ]:
# 1) Setup imports and module path
import sys
from pathlib import Path

ROOT = Path.cwd()
V2_ROOT = ROOT / 'fyp_finance_ml_v2'
sys.path.insert(0, str(V2_ROOT))

print('ROOT:', ROOT)
print('V2_ROOT exists:', V2_ROOT.exists())


In [ ]:
# 2) Import your friend's modules
from src.config import AppConfig
from src.data_loader import load_data
from src.features import build_feature_frame, feature_columns_for_set
from src.labels import add_forward_labels
from src.splits import time_split
from src.models import build_models, choose_threshold
from src.leakage import leakage_guard
from src.evaluation import compute_ml_metrics, compute_signal_metrics
from src.backtest import run_top_k_backtest, run_benchmark_buy_hold, relative_summary

import numpy as np
import pandas as pd


In [ ]:
# 3) Configure run mode here
cfg = AppConfig()
MODE = 'live'      # change to 'synthetic' if needed
NOTEBOOK_TAG = 'nb_step'

# Optional: quicker debug run
# cfg.horizons = [1]
# cfg.feature_sets = {'F1_momentum': cfg.feature_sets['F1_momentum']}

print('Mode:', MODE)
print('Horizons:', cfg.horizons)
print('Feature sets:', list(cfg.feature_sets.keys()))


In [ ]:
# 4) Load data
prices, benchmark, macro, fundamentals = load_data(cfg, mode=MODE)
prices = prices.loc[prices['ticker'].isin(cfg.tickers)].copy()

print('prices shape:', prices.shape)
print('benchmark shape:', benchmark.shape)
print('macro shape:', macro.shape)
print('fundamentals shape:', fundamentals.shape)
print('tickers loaded:', prices['ticker'].nunique() if not prices.empty else 0)


In [ ]:
# 5) Build feature frame + labels
feature_df = build_feature_frame(prices, benchmark, macro, fundamentals)
full_df = add_forward_labels(feature_df, cfg.horizons)

print('feature_df shape:', feature_df.shape)
print('full_df shape:', full_df.shape)
full_df.head()


In [ ]:
# 6) Single-block debug runner (pick one feature set + horizon first)
feature_set_name = 'F5_full_finance'
horizon = 3

groups = cfg.feature_sets[feature_set_name]
feat_cols = feature_columns_for_set(groups)
leakage_guard(feat_cols)

label_col = f'label_{horizon}d'
ret_col = f'fwd_ret_{horizon}d'
keep_cols = list(dict.fromkeys(['date','ticker','ret_5',label_col,ret_col] + feat_cols))
work = full_df[keep_cols].copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col, ret_col])

train, val, test, split_meta = time_split(work, cfg.train_frac, cfg.val_frac)
valid_feat_cols = [c for c in feat_cols if c in train.columns and train[c].notna().any()]

X_train, y_train = train[valid_feat_cols], train[label_col].astype(int)
X_val, y_val = val[valid_feat_cols], val[label_col].astype(int)
X_test, y_test = test[valid_feat_cols], test[label_col].astype(int)

print('split:', split_meta)
print('train/val/test:', X_train.shape, X_val.shape, X_test.shape)


In [ ]:
# 7) Train models + evaluate this selected block
models = build_models(cfg.random_seed, use_xgboost=cfg.use_xgboost)
rows = []

bench_daily, _ = run_benchmark_buy_hold(benchmark, horizon=horizon)

for model_name, model in models.items():
    model.fit(X_train, y_train)
    val_proba = model.predict_proba(X_val)[:, 1]
    thr, val_bal = choose_threshold(y_val, val_proba)

    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= thr).astype(int)

    scored = test[['date','ticker','ret_5',ret_col]].copy()
    scored['score'] = test_proba

    ml = compute_ml_metrics(y_test, test_pred, test_proba)
    sig = compute_signal_metrics(scored, 'score', ret_col, cfg.top_k, n_buckets=cfg.n_deciles)
    bt_daily, bt_sum = run_top_k_backtest(scored, 'score', ret_col, top_k=cfg.top_k, transaction_cost_bps=cfg.transaction_cost_bps)
    rel = relative_summary(bt_daily, bench_daily)

    rows.append({
        'feature_set': feature_set_name, 'horizon': horizon, 'model': model_name,
        'threshold': thr, 'val_bal_acc': val_bal, **ml, **sig, **bt_sum, **rel
    })

pd.DataFrame(rows).sort_values(['balanced_accuracy','sharpe'], ascending=False)


In [ ]:
# 8) Full loop run (optional, longer)
run_full = False
all_rows = []

if run_full:
    models = build_models(cfg.random_seed, use_xgboost=cfg.use_xgboost)
    for h in cfg.horizons:
        label_col = f'label_{h}d'
        ret_col = f'fwd_ret_{h}d'
        bench_daily, _ = run_benchmark_buy_hold(benchmark, horizon=h)

        for fs_name, groups in cfg.feature_sets.items():
            feat_cols = feature_columns_for_set(groups)
            leakage_guard(feat_cols)
            keep_cols = list(dict.fromkeys(['date','ticker','ret_5',label_col,ret_col] + feat_cols))
            work = full_df[keep_cols].copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col, ret_col])
            train, val, test, split_meta = time_split(work, cfg.train_frac, cfg.val_frac)
            valid_feat_cols = [c for c in feat_cols if c in train.columns and train[c].notna().any()]

            X_train, y_train = train[valid_feat_cols], train[label_col].astype(int)
            X_val, y_val = val[valid_feat_cols], val[label_col].astype(int)
            X_test, y_test = test[valid_feat_cols], test[label_col].astype(int)

            if X_train.empty or X_val.empty or X_test.empty:
                continue

            for model_name, model in models.items():
                model.fit(X_train, y_train)
                val_proba = model.predict_proba(X_val)[:, 1]
                thr, val_bal = choose_threshold(y_val, val_proba)
                test_proba = model.predict_proba(X_test)[:, 1]
                test_pred = (test_proba >= thr).astype(int)

                scored = test[['date','ticker','ret_5',ret_col]].copy()
                scored['score'] = test_proba

                ml = compute_ml_metrics(y_test, test_pred, test_proba)
                sig = compute_signal_metrics(scored, 'score', ret_col, cfg.top_k, n_buckets=cfg.n_deciles)
                bt_daily, bt_sum = run_top_k_backtest(scored, 'score', ret_col, top_k=cfg.top_k, transaction_cost_bps=cfg.transaction_cost_bps)
                rel = relative_summary(bt_daily, bench_daily)

                all_rows.append({
                    'feature_set': fs_name, 'horizon': h, 'model': model_name,
                    'threshold': thr, 'val_bal_acc': val_bal, **ml, **sig, **bt_sum, **rel
                })

summary = pd.DataFrame(all_rows)
if not summary.empty:
    out_dir = ROOT / 'outputs' / 'notebook_step'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{NOTEBOOK_TAG}_{MODE}_summary.csv'
    summary.to_csv(out_path, index=False)
    print('Saved:', out_path)
summary.head()
